# Export Landsat 7 patches — 476 clusters Niger

Lance les 476 exports en **une seule exécution**.
Les exports tournent sur les serveurs Google — tu peux fermer l'onglet.

Prérequis :
- GEE Asset : `projects/balmy-cab-498510-j1/assets/gee_clusters` (déjà créé)

In [ ]:
# Authentification GEE
import ee
try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()
print('GEE OK')

In [ ]:
ASSET_ID    = 'projects/balmy-cab-498510-j1/assets/gee_clusters'
DRIVE_FOLDER = 'poverty_sae_patches'
SCALE       = 30
RADIUS      = 3360
START       = '2011-01-01'
END         = '2012-12-31'
MAX_CLOUD   = 30
START_IDX   = 0     # 0 = premier
END_IDX     = 476   # 476 = dernier

# Pour tester avec 10 clusters : START_IDX=0, END_IDX=10

In [ ]:
clusters = ee.FeatureCollection(ASSET_ID)
features = clusters.toList(clusters.size()).getInfo()
print(f'Total clusters dans l\'asset : {len(features)}')
print(f'À exporter : indices {START_IDX} à {END_IDX-1}')

# Tranche (test ou complet)
selected = features[START_IDX:END_IDX]
print(f'Soit {len(selected)} clusters')

# Aperçu : premier et dernier
premier = selected[0]['properties']['cluster']
dernier = selected[-1]['properties']['cluster']
print(f'Premier : {premier}, Dernier : {dernier}')

In [ ]:
def export_patch(f, idx, total):
    cid_raw = f['properties']['cluster']
    # Gère cluster entier ou string
    cid_str = str(cid_raw).zfill(3) if isinstance(cid_raw, int) else str(cid_raw)
    lon = f['geometry']['coordinates'][0]
    lat = f['geometry']['coordinates'][1]

    pt  = ee.Geometry.Point([lon, lat])
    box = pt.buffer(RADIUS).bounds()

    # Collection Landsat 7 : toutes les scènes de la zone
    collection = ee.ImageCollection('LANDSAT/LE07/C02/T1_L2') \
        .filterDate(START, END) \
        .filterBounds(box) \
        .filter(ee.Filter.lt('CLOUD_COVER', MAX_CLOUD))

    # Composite median (fusionne toutes les dates)
    composite = collection \
        .select(['SR_B3', 'SR_B2', 'SR_B1']) \
        .median() \
        .clip(box)

    # PAS de reproject ici : l'export gère scale/crs
    # PAS de getInfo() ici : éviterait rate limiting

    task = ee.batch.Export.image.toDrive(
        image=composite,
        description=f'cluster_{cid_str}',
        fileNamePrefix=f'cluster_{cid_str}',
        folder=DRIVE_FOLDER,
        scale=SCALE,
        crs='EPSG:4326',
        maxPixels=1e9,
        fileFormat='GEO_TIFF'
    )

    task.start()
    print(f'  [{idx+1}/{total}] cid={cid_str} ({lon:.2f}, {lat:.2f})')
    return task

In [ ]:
print(f'Lancement de {len(selected)} exports...')

tasks = {}
failures = []

for i, f in enumerate(selected):
    cid_raw = f['properties']['cluster']
    cid_str = str(cid_raw).zfill(3) if isinstance(cid_raw, int) else str(cid_raw)
    try:
        t = export_patch(f, i, len(selected))
        tasks[cid_str] = t
    except Exception as e:
        print(f'  ERREUR cid={cid_str} : {type(e).__name__}: {e}')
        failures.append(cid_str)

print(f'\n=== RÉSUMÉ ===')
print(f'  Lancés : {len(tasks)}')
print(f'  Échecs : {len(failures)}')
if failures:
    print(f'  Liste : {failures}')

## Surveiller l'avancement

Exécute la cellule ci-dessous pour voir le statut actuel.
Tu peux aussi vérifier sur https://code.earthengine.google.com → Tasks (panneau droit)

Chaque export prend ~15-60 secondes. Les 476 exports → **1 à 2 heures**.

In [ ]:
from collections import Counter

statuses = Counter(t.status()['state'] for t in tasks.values())
print(dict(statuses))

# Afficher les IDs des échecs si présents
failed_ids = [k for k, t in tasks.items() if t.status()['state'] == 'FAILED']
if failed_ids:
    print(f'Cluster en échec : {failed_ids}\. Peut réessayer en les relançant')

## Relancer les échecs (si nécessaire)

S'il y a des FAILED, exécute cette cellule pour les relancer.

In [ ]:
failed = {k: t for k, t in tasks.items() if t.status()['state'] == 'FAILED'}
if failed:
    print(f'Relance de {len(failed)} échecs...')
    for k, t in failed.items():
        # TODO : ré-exporter ce cluster précis
        pass
else:
    print('Aucun échec à relancer.')

## Téléchargement

Quand tout est COMPLETED :
1. https://drive.google.com → dossier `poverty_sae_patches/`
2. Ctrl+A → clic droit → Télécharger
3. Décompresser et copier dans `data/processed/patches_landsat/`